In [ ]:
!nvidia-smi

# Qwen 2.5 7B: mean-ablation patching + C4 flagging + temperature scaling

Model: Qwen/Qwen2.5-7B | [HF](https://huggingface.co/Qwen/Qwen2.5-7B)\
GPU: H100 SXM 80GB | Layers 0-18 ablation | Composition tests | Temp scaling | C4 flagging | Date: 2026-04-07

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

In [ ]:
!pip install transformers datasets scipy huggingface_hub -q

try:
    from google.colab import userdata
    import os
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("Not on Colab or no HF_TOKEN secret")

In [ ]:
# Pre-download models (uses HF_TOKEN from env)
!hf download Qwen/Qwen2.5-7B


In [ ]:
import gc
import json

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
def _get_layer_modules(model, layer_idx):
    if hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        block = model.transformer.h[layer_idx]
        return block.attn, block.mlp
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        block = model.model.layers[layer_idx]
        return block.self_attn, block.mlp
    raise ValueError(f'Unsupported architecture: {type(model).__name__}')

def load_wikitext(split='test', max_docs=None):
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split=split)
    docs, current = [], []
    for row in ds:
        text = row['text']
        if text.strip() == '' and current:
            docs.append('\n'.join(current)); current = []
            if max_docs and len(docs) >= max_docs: break
        elif text.strip(): current.append(text)
    if current: docs.append('\n'.join(current))
    return docs

def compute_loss_residuals(losses, max_softmax, activation_norm):
    X = np.column_stack([max_softmax, activation_norm, np.ones(len(losses))])
    beta, _, _, _ = np.linalg.lstsq(X, losses, rcond=None)
    return losses - X @ beta

def collect_layer_data(model, tokenizer, docs, layer, device, max_tokens=200000, max_length=512):
    model.eval()
    all_acts, all_losses, all_softmax, all_norms = [], [], [], []
    total = 0
    for doc in docs:
        if total >= max_tokens: break
        if not doc.strip(): continue
        tokens = tokenizer(doc, return_tensors='pt', truncation=True, max_length=max_length)
        input_ids = tokens['input_ids'].to(device)
        if input_ids.size(1) < 2: continue
        with torch.inference_mode():
            outputs = model(input_ids, output_hidden_states=True)
        h = outputs.hidden_states[layer + 1][0, :-1, :].cpu()
        logits = outputs.logits[0, :-1, :]
        labels = input_ids[0, 1:]
        losses = F.cross_entropy(logits, labels, reduction='none').cpu()
        sm = F.softmax(logits, dim=-1).max(dim=-1).values.cpu()
        norms = h.norm(dim=-1)
        all_acts.append(h); all_losses.append(losses); all_softmax.append(sm); all_norms.append(norms)
        total += h.size(0)
    print(f'    {total} positions from {len(all_acts)} documents')
    return {
        'activations': torch.cat(all_acts).float(),
        'losses': torch.cat(all_losses).float().numpy(),
        'max_softmax': torch.cat(all_softmax).float().numpy(),
        'activation_norm': torch.cat(all_norms).float().numpy(),
    }

def train_linear_binary(train_data, seed=42, epochs=20, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    acts = train_data['activations']
    residuals = compute_loss_residuals(
        train_data['losses'], train_data['max_softmax'], train_data['activation_norm']
    )
    targets = torch.from_numpy((residuals > 0).astype(np.float32))
    head = torch.nn.Linear(acts.size(1), 1)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(acts, targets)
    dl = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=True)
    head.train()
    for _ in range(epochs):
        for bx, by in dl:
            loss = F.binary_cross_entropy_with_logits(head(bx).squeeze(-1), by)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return head

## Load model and data

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-7B'
PEAK_LAYER = 18
EVAL_BUDGET = 15000

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16,
    attn_implementation='sdpa'
).cuda()
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
N_PARAMS = sum(p.numel() for p in model.parameters()) / 1e9
print(f'{N_PARAMS:.1f}B params, {N_LAYERS} layers, {HIDDEN_DIM} dim')

MAX_TRAIN = max(150 * HIDDEN_DIM, 200000)

attn_mod, mlp_mod = _get_layer_modules(model, 0)
print(f'Hooks: {type(attn_mod).__name__}, {type(mlp_mod).__name__}')

In [ ]:
wiki_train = load_wikitext('train', max_docs=2000)
wiki_test = load_wikitext('test', max_docs=500)

## Train observer

In [ ]:
train_data = collect_layer_data(model, tokenizer, wiki_train, PEAK_LAYER, 'cuda', MAX_TRAIN)
observer = train_linear_binary(train_data, seed=42)
observer.eval()
del train_data; gc.collect(); torch.cuda.empty_cache()

## Compute component means

In [ ]:
eval_docs = wiki_test[:200]
accumulators = {}

def make_accumulator_hook(key):
    def hook_fn(module, inp, out):
        val = out[0] if isinstance(out, tuple) else out
        if key not in accumulators: accumulators[key] = []
        accumulators[key].append(val.detach().cpu())
        return out
    return hook_fn

handles = []
for layer in range(PEAK_LAYER + 1):
    attn, mlp = _get_layer_modules(model, layer)
    handles.append(attn.register_forward_hook(make_accumulator_hook((layer, 'attn'))))
    handles.append(mlp.register_forward_hook(make_accumulator_hook((layer, 'mlp'))))

total = 0
with torch.inference_mode():
    for doc in eval_docs:
        if total >= EVAL_BUDGET: break
        if not doc.strip(): continue
        input_ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].cuda()
        if input_ids.size(1) < 2: continue
        model(input_ids)
        total += input_ids.size(1) - 1

for h in handles: h.remove()

component_means = {}
for key, tensors in accumulators.items():
    component_means[key] = torch.cat(tensors, dim=1).mean(dim=1, keepdim=True)

del accumulators; gc.collect()
print(f'{len(component_means)} components, {total} positions')

## Baseline

In [ ]:
def collect_baseline(model, tokenizer, observer, peak_layer, docs, device, max_tokens):
    all_obs, all_loss, all_sm = [], [], []
    total = 0
    model.eval(); observer.eval()
    with torch.inference_mode():
        for doc in docs:
            if total >= max_tokens: break
            if not doc.strip(): continue
            input_ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].to(device)
            if input_ids.size(1) < 2: continue
            outputs = model(input_ids, output_hidden_states=True)
            h = outputs.hidden_states[peak_layer + 1][0, :-1, :].cpu().float()
            logits = outputs.logits[0, :-1, :]
            losses = F.cross_entropy(logits, input_ids[0, 1:], reduction='none').cpu().float().numpy()
            sm = F.softmax(logits, dim=-1).max(dim=-1).values.cpu().float().numpy()
            all_obs.append(observer(h).squeeze(-1).numpy())
            all_loss.append(losses); all_sm.append(sm)
            total += h.size(0)
    return {
        'obs_scores': np.concatenate(all_obs)[:max_tokens],
        'losses': np.concatenate(all_loss)[:max_tokens],
        'max_softmax': np.concatenate(all_sm)[:max_tokens],
        'n': min(total, max_tokens),
    }

baseline = collect_baseline(model, tokenizer, observer, PEAK_LAYER, eval_docs, 'cuda', EVAL_BUDGET)
n_eval = baseline['n']
print(f'{n_eval} positions')

## Part 1: Per-component mean ablation

In [ ]:
def residualized_delta(base_obs, patched_obs, base_sm, patched_sm):
    sm_delta = patched_sm - base_sm
    obs_delta = patched_obs - base_obs
    if np.std(sm_delta) > 1e-8:
        beta = np.cov(obs_delta, sm_delta)[0, 1] / np.var(sm_delta)
        return float((obs_delta - beta * sm_delta).mean())
    return float(obs_delta.mean())

def mean_ablate(model, tokenizer, observer, peak_layer, docs, device,
                max_tokens, target_layer, component, means):
    mean_val = means[(target_layer, component)]
    attn, mlp = _get_layer_modules(model, target_layer)
    target = attn if component == 'attn' else mlp
    def hook_fn(module, inp, out):
        m = mean_val.to(out[0].device if isinstance(out, tuple) else out.device)
        expanded = m.expand_as(out[0] if isinstance(out, tuple) else out)
        if isinstance(out, tuple): return (expanded,) + out[1:]
        return expanded
    handle = target.register_forward_hook(hook_fn)
    result = collect_baseline(model, tokenizer, observer, peak_layer, docs, device, max_tokens)
    handle.remove()
    return result

print(f'{"Layer":<8} {"Comp":<6} {"Obs resid":>16} {"Loss":>12} {"Raw obs":>15}')
print('-' * 59)

ablation_results = {}
for layer in range(PEAK_LAYER + 1):
    layer_r = {}
    for comp in ['attn', 'mlp']:
        p = mean_ablate(model, tokenizer, observer, PEAK_LAYER, eval_docs, 'cuda',
                        EVAL_BUDGET, layer, comp, component_means)
        obs_r = residualized_delta(baseline['obs_scores'], p['obs_scores'], baseline['max_softmax'], p['max_softmax'])
        loss_d = float(p['losses'].mean() - baseline['losses'].mean())
        raw = float(p['obs_scores'].mean() - baseline['obs_scores'].mean())
        layer_r[comp] = {'obs_resid_delta': obs_r, 'loss_delta': loss_d, 'raw_obs_delta': raw}
        print(f'{layer:<8} {comp:<6} {obs_r:>+16.4f} {loss_d:>+12.4f} {raw:>+15.4f}')
    ablation_results[layer] = layer_r

## Part 2: Composition tests

In [ ]:
def mean_ablate_group(model, tokenizer, observer, peak_layer, docs, device,
                      max_tokens, components, means):
    handles = []
    for layer, comp in components:
        mv = means[(layer, comp)]
        attn, mlp = _get_layer_modules(model, layer)
        target = attn if comp == 'attn' else mlp
        def make_hook(mean_val):
            def hook_fn(module, inp, out):
                m = mean_val.to(out[0].device if isinstance(out, tuple) else out.device)
                expanded = m.expand_as(out[0] if isinstance(out, tuple) else out)
                if isinstance(out, tuple): return (expanded,) + out[1:]
                return expanded
            return hook_fn
        handles.append(target.register_forward_hook(make_hook(mv)))
    result = collect_baseline(model, tokenizer, observer, peak_layer, docs, device, max_tokens)
    for h in handles: h.remove()
    return result

attn_ranked = sorted(
    [(l, ablation_results[l]['attn']['obs_resid_delta']) for l in range(PEAK_LAYER + 1)],
    key=lambda x: abs(x[1]), reverse=True)
mlp_ranked = sorted(
    [(l, ablation_results[l]['mlp']['obs_resid_delta']) for l in range(PEAK_LAYER + 1)],
    key=lambda x: abs(x[1]), reverse=True)

top2_attn = sorted([attn_ranked[0][0], attn_ranked[1][0]])
top4_attn = sorted([r[0] for r in attn_ranked[:4]])
top2_mlp = sorted([mlp_ranked[0][0], mlp_ranked[1][0]])

groups = {
    f'attn_{top2_attn[0]}_{top2_attn[1]}': [(l, 'attn') for l in top2_attn],
    'attn_top4': [(l, 'attn') for l in top4_attn],
    f'mlp_{top2_mlp[0]}_{top2_mlp[1]}': [(l, 'mlp') for l in top2_mlp],
    'all_top': [(l, 'attn') for l in top4_attn] + [(l, 'mlp') for l in top2_mlp],
}

composition_results = {}
for gname, components in groups.items():
    p = mean_ablate_group(model, tokenizer, observer, PEAK_LAYER, eval_docs, 'cuda',
                          EVAL_BUDGET, components, component_means)
    obs_r = residualized_delta(baseline['obs_scores'], p['obs_scores'], baseline['max_softmax'], p['max_softmax'])
    loss_d = float(p['losses'].mean() - baseline['losses'].mean())
    expected = sum(ablation_results[l][c]['obs_resid_delta'] for l, c in components)
    composition_results[gname] = {
        'obs_resid_delta': obs_r, 'loss_delta': loss_d,
        'expected_additive': expected, 'interaction': obs_r - expected,
        'components': [[l, c] for l, c in components],
    }
    tag = 'additive' if abs(obs_r - expected) < 0.01 else f'interaction={obs_r - expected:+.4f}'
    print(f'{gname:<22}: obs_resid={obs_r:+.4f}  expected={expected:+.4f}  ({tag})')

## Summary

In [ ]:
top_attn_layers = [r[0] for r in attn_ranked[:3]]
depth_range = f'{min(top_attn_layers)/N_LAYERS:.2f}-{max(top_attn_layers)/N_LAYERS:.2f}'

print(f'{"Layer":<8} {"Attn":>15} {"MLP":>15} {"Attn loss":>11} {"MLP loss":>11}')
print('-' * 62)
for layer in range(PEAK_LAYER + 1):
    ar = ablation_results[layer]
    print(f'{layer:<8} {ar["attn"]["obs_resid_delta"]:>+15.4f} {ar["mlp"]["obs_resid_delta"]:>+15.4f}'
          f' {ar["attn"]["loss_delta"]:>+11.4f} {ar["mlp"]["loss_delta"]:>+11.4f}')

print(f'\nTop 3 attn: {top_attn_layers} (depth: {depth_range})')
for gname, vals in composition_results.items():
    tag = 'additive' if abs(vals['interaction']) < 0.01 else f'interaction={vals["interaction"]:+.4f}'
    print(f'  {gname:<22}: {vals["obs_resid_delta"]:+.4f} ({tag})')

## Temperature scaling control

In [ ]:
from scipy.stats import pearsonr, rankdata
from scipy.optimize import minimize_scalar

def partial_spearman(x, y, covariates):
    rx, ry = rankdata(x), rankdata(y)
    rc = np.column_stack([rankdata(c) for c in covariates])
    rc = np.column_stack([rc, np.ones(len(rc))])
    coef_x = np.linalg.lstsq(rc, rx, rcond=None)[0]
    coef_y = np.linalg.lstsq(rc, ry, rcond=None)[0]
    r, p = pearsonr(rx - rc @ coef_x, ry - rc @ coef_y)
    return float(r), float(p)

MAX_TEST = MAX_TRAIN // 2

all_val_logits, all_val_labels = [], []
total_val = 0
model.eval()
with torch.inference_mode():
    for doc in load_wikitext('validation', max_docs=500):
        if total_val >= 50000: break
        if not doc.strip(): continue
        input_ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].cuda()
        if input_ids.size(1) < 2: continue
        logits = model(input_ids).logits[0, :-1, :].cpu().float()
        labels = input_ids[0, 1:].cpu()
        all_val_logits.append(logits); all_val_labels.append(labels)
        total_val += logits.size(0)

val_logits = torch.cat(all_val_logits)[:50000]
val_labels = torch.cat(all_val_labels)[:50000]

def nll_at_temp(T):
    return F.cross_entropy(val_logits / T, val_labels).item()

res = minimize_scalar(nll_at_temp, bounds=(0.1, 5.0), method='bounded')
T_opt = res.x
print(f'Optimal temperature: {T_opt:.4f}')
print(f'NLL raw: {nll_at_temp(1.0):.4f}, NLL calibrated: {nll_at_temp(T_opt):.4f}')
del all_val_logits, all_val_labels, val_logits, val_labels

all_acts, all_losses, all_sm_raw, all_sm_cal, all_norms = [], [], [], [], []
total_t = 0
with torch.inference_mode():
    for doc in wiki_test[:500]:
        if total_t >= MAX_TEST: break
        if not doc.strip(): continue
        input_ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].cuda()
        if input_ids.size(1) < 2: continue
        outputs = model(input_ids, output_hidden_states=True)
        h = outputs.hidden_states[PEAK_LAYER + 1][0, :-1, :].cpu().float()
        logits = outputs.logits[0, :-1, :]
        labels = input_ids[0, 1:]
        losses = F.cross_entropy(logits, labels, reduction='none').cpu().float().numpy()
        sm_raw = F.softmax(logits, dim=-1).max(dim=-1).values.cpu().float().numpy()
        sm_cal = F.softmax(logits / T_opt, dim=-1).max(dim=-1).values.cpu().float().numpy()
        norms = h.norm(dim=-1).numpy()
        all_acts.append(h); all_losses.append(losses)
        all_sm_raw.append(sm_raw); all_sm_cal.append(sm_cal); all_norms.append(norms)
        total_t += h.size(0)

n_t = min(total_t, MAX_TEST)
test_acts = torch.cat(all_acts)[:n_t]
test_losses = np.concatenate(all_losses)[:n_t]
test_sm_raw = np.concatenate(all_sm_raw)[:n_t]
test_sm_cal = np.concatenate(all_sm_cal)[:n_t]
test_norms = np.concatenate(all_norms)[:n_t]
print(f'{n_t} test positions')

# Train observer
head_ts = train_linear_binary({
    'activations': test_acts, 'losses': test_losses,
    'max_softmax': test_sm_raw, 'activation_norm': test_norms,
}, seed=42)
head_ts.eval()
with torch.inference_mode():
    obs_sc = head_ts(test_acts).squeeze(-1).numpy()

rho_raw, _ = partial_spearman(obs_sc, test_losses, [test_sm_raw, test_norms])
rho_cal, _ = partial_spearman(obs_sc, test_losses, [test_sm_cal, test_norms])

temp_scaling_result = {
    'optimal_temperature': float(T_opt),
    'partial_corr_raw_confidence': float(rho_raw),
    'partial_corr_calibrated_confidence': float(rho_cal),
    'delta': float(rho_cal - rho_raw),
}

print(f'Partial corr (raw confidence):        {rho_raw:+.4f}')
print(f'Partial corr (calibrated T={T_opt:.2f}): {rho_cal:+.4f}')
print(f'Delta: {rho_cal - rho_raw:+.4f}')

del all_acts, all_losses, all_sm_raw, all_sm_cal, all_norms, test_acts
gc.collect(); torch.cuda.empty_cache()

## C4 cross-domain flagging

In [ ]:
from datasets import load_dataset

c4_docs = []
ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
for i, row in enumerate(ds):
    if i < 50000: continue
    text = row['text'].strip()
    if len(text) > 100: c4_docs.append(text)
    if len(c4_docs) >= 500: break
print(f'{len(c4_docs)} C4 test docs')

output_layer = N_LAYERS - 1
MAX_TEST = MAX_TRAIN // 2

c4_peak = collect_layer_data(model, tokenizer, c4_docs, PEAK_LAYER, 'cuda', MAX_TEST)
c4_output = collect_layer_data(model, tokenizer, c4_docs, output_layer, 'cuda', MAX_TEST)

wiki_test_peak = collect_layer_data(model, tokenizer, wiki_test, PEAK_LAYER, 'cuda', MAX_TEST)
wiki_test_output = collect_layer_data(model, tokenizer, wiki_test, output_layer, 'cuda', MAX_TEST)

seeds = [42, 43, 44]
flag_rates = [0.05, 0.10, 0.20, 0.30]

def run_flagging(train_peak, test_peak, test_output, seeds, flag_rates, label):
    n_test = min(len(test_peak['losses']), len(test_output['losses']))
    losses = test_peak['losses'][:n_test]
    acts = test_peak['activations'][:n_test]
    output_sm = test_output['max_softmax'][:n_test]
    median_loss = float(np.median(losses))
    is_high_loss = losses > median_loss

    print(f'\n{label}: {n_test} tokens, median loss = {median_loss:.4f}')
    all_seed = []
    for seed in seeds:
        head = train_linear_binary(train_peak, seed=seed)
        head.eval()
        with torch.inference_mode():
            obs_scores = head(acts).squeeze(-1).numpy()
        seed_r = {}
        for rate in flag_rates:
            k = int(n_test * rate)
            obs_flagged = obs_scores >= np.sort(obs_scores)[-k]
            conf_flagged = output_sm <= np.sort(output_sm)[k]
            obs_excl = int((obs_flagged & ~conf_flagged & is_high_loss).sum())
            conf_excl = int((conf_flagged & ~obs_flagged & is_high_loss).sum())
            obs_prec = float(is_high_loss[obs_flagged].mean()) if obs_flagged.sum() > 0 else 0.0
            conf_prec = float(is_high_loss[conf_flagged].mean()) if conf_flagged.sum() > 0 else 0.0
            seed_r[str(rate)] = {'obs_prec': obs_prec, 'conf_prec': conf_prec,
                                  'obs_exclusive': obs_excl, 'conf_exclusive': conf_excl}
        all_seed.append(seed_r)

    summary = {}
    for rate in flag_rates:
        r = str(rate)
        summary[r] = {
            'observer_precision': float(np.mean([s[r]['obs_prec'] for s in all_seed])),
            'confidence_precision': float(np.mean([s[r]['conf_prec'] for s in all_seed])),
            'observer_exclusive': float(np.mean([s[r]['obs_exclusive'] for s in all_seed])),
            'confidence_exclusive': float(np.mean([s[r]['conf_exclusive'] for s in all_seed])),
        }
        print(f'  {rate:.0%}: obs_prec={summary[r]["observer_precision"]:.3f}  '
              f'conf_prec={summary[r]["confidence_precision"]:.3f}  '
              f'obs_excl={summary[r]["observer_exclusive"]:.0f}')

    return {'n_test': n_test, 'median_loss': median_loss, 'per_seed': all_seed, 'summary': summary}

wiki_train_peak = collect_layer_data(model, tokenizer, wiki_train, PEAK_LAYER, 'cuda', MAX_TRAIN)

c4_flagging = run_flagging(wiki_train_peak, c4_peak, c4_output, seeds, flag_rates, 'C4 (WikiText-trained)')
wiki_flagging = run_flagging(wiki_train_peak, wiki_test_peak, wiki_test_output, seeds, flag_rates, 'WikiText (reference)')

del wiki_train_peak, c4_peak, c4_output, wiki_test_peak, wiki_test_output
gc.collect(); torch.cuda.empty_cache()

## Save

In [ ]:
output = {
    'model': MODEL_ID,
    'n_params_b': round(N_PARAMS, 1),
    'n_layers': N_LAYERS,
    'hidden_dim': HIDDEN_DIM,
    'peak_layer': PEAK_LAYER,
    'n_eval': n_eval,
    'ablation_results': {str(k): v for k, v in ablation_results.items()},
    'composition_results': composition_results,
    'top_attn_layers': top_attn_layers,
    'top_attn_depth_range': depth_range,
    'temperature_scaling': temp_scaling_result,
    'c4_flagging': c4_flagging,
    'wikitext_flagging': wiki_flagging,
}

with open('mechanistic_7b.json', 'w') as f:
    json.dump(output, f, indent=2)
print(json.dumps(output, indent=2))

In [ ]:
from google.colab import files
files.download('mechanistic_7b.json')